# 04 · Gold con Change Data Feed

⏱ **12 min** · Slide: 19 (CDF en un pipeline medallón)

**Objetivo:** que gold recalcule **solo las redes y días afectados** por los cambios de silver.

```
silver_episodes ──► table_changes(silver) ──► claves (red, día) afectadas ──► recalcular esas claves ──► MERGE ──► gold_network_daily
```

| Paso | Qué haces |
|---|---|
| 1 | Ver los cambios pendientes de silver |
| 2 | Construir gold (CDF + MERGE) |
| 3 | Comprobar contra un cálculo completo |
| 4 | Cadena completa con un día nuevo |

**Requiere:** notebook 03.

In [0]:
%run ./00_Setup

## Paso 1 · Cambios pendientes de silver y claves afectadas

In [0]:
desde, hasta = last_processed("gold") + 1, table_version(SILVER)
cambios = read_changes(SILVER, desde, hasta)

print(f"Versiones pendientes de silver: {desde} a {hasta} | filas de cambio: {cambios.count()}")
display(cambios.select(*GOLD_KEYS).distinct().orderBy(*GOLD_KEYS))

👀 **Qué observar:** de todos los cambios solo importan las claves `(airdate, network_name)` que tocaron.
La primera vez son todas (gold no existe); después serán pocas.

## Paso 2 · Construir gold con CDF + MERGE

In [0]:
print(inspect.getsource(build_gold))

In [0]:
build_gold()

In [0]:
display(spark.table(GOLD).orderBy(F.col("episodes").desc()).limit(10))

In [0]:
build_gold()   # segunda vez: nada que hacer

## Paso 3 · Comprobar contra un cálculo completo

Gold incremental debe dar **exactamente** lo mismo que agregar toda la tabla silver.

In [0]:
completo = spark.table(SILVER).groupBy(*GOLD_KEYS).agg(F.count("*").alias("episodes"))

diferencias = (completo.alias("c")
               .join(spark.table(GOLD).alias("g"),
                     F.expr("c.airdate <=> g.airdate AND c.network_name <=> g.network_name"), "full")
               .where("g.episodes IS NULL OR c.episodes IS NULL OR g.episodes <> c.episodes"))

print("Claves en gold       :", spark.table(GOLD).count())
print("Claves esperadas     :", completo.count())
print("Claves con diferencia:", diferencias.count(), "← debe ser 0")

## Paso 4 · Cadena completa con un día nuevo

RAW → bronze → silver → gold, cada capa procesa solo lo nuevo.

In [0]:
download_days(next_day(), 1)
run_bronze()
build_silver()
build_gold()

In [0]:
print("Claves (red, día) en gold:", spark.table(GOLD).count())
display(spark.table(GOLD).orderBy(F.col("airdate").desc(), F.col("episodes").desc()).limit(10))

👀 **Qué observar:** gold recalculó solo las claves del día nuevo (mira el mensaje de `build_gold()`), no las decenas de
claves que ya tenía. Con millones de filas, esa es la diferencia entre minutos y segundos.

## ✔ Checkpoint

- [ ] Entiendo por qué gold lee el CDF de silver y no toda la tabla
- [ ] Gold incremental coincide con el cálculo completo (0 diferencias)
- [ ] Ejecuté la cadena completa con un día nuevo

**Siguiente:** `05_Time_Travel_y_Mantenimiento`